In [30]:
 !pip install pymupdf
 !pip install pdfplumber
 !pip install faiss-cpu

In [1]:
from util import StructuredDataLoader, UnstructuredDataLoader

In [3]:
st_loader = StructuredDataLoader()

file_path = "amazon_reviews.csv"
data = st_loader.load_csv(file_path)

data.head()

,product_name,product_id,price,description,reviewer_name,rating,review_title,review_body,review_date
0,Stock process,G7GD90FUAD,64.99,Think type good coach part. Medical herself me...,Richard Reilly,5,Sort spring medical already,Movie reality service throughout everybody sor...,2022-01-24
1,Area skin president hair upon,SYSFMJBX59,281.87,Trouble answer for major find. Only country ed...,Sean Chase,1,Take drive money hope door reduce behind,Before issue reflect will. Little pay change o...,2021-04-25
2,Well on grow business good indeed measure,EMBQF60QVR,475.05,You off cold rest course little concern. Witho...,Michael Ritter,4,Heavy beautiful still party rock boy easy succ...,Similar page recently commercial system employ...,2021-01-25
3,As item bit old dark,42830GWDNS,64.96,Material condition out shake. Everybody hotel ...,Cynthia Stewart,5,Suddenly democratic really high thousand seem ...,Really possible carry hold pick stay goal. Det...,2024-10-30
4,Myself agreement reveal owner why myself,3BFRSTIFDF,474.06,Town with seem follow most option sea. Own coa...,Michael Underwood,3,Nation just fear human international,Describe fact film entire card him. Too item s...,2020-11-25


In [4]:
unstr_loader = UnstructuredDataLoader()

In [5]:
text = unstr_loader.extract_text_from_pdf("Table_Reconstruction.pdf")

In [35]:
# text

In [6]:
tables_md, tables_df = unstr_loader.extract_tables_from_pdf("Table_Reconstruction.pdf")

In [7]:
tables_md[2]

'--- Table 1 on Page 8 ---\n| **Method** | **IoU** | **FinTabNet** | **ICDAR-13** | **Sci-TSR** | **TUCD** |\n| --- | --- | --- | --- | --- | --- |\n|  |  | TSR-F1↑ | TSR-F1↑ | TSR-F1↑ | TSR-F1↑ |\n| TS-Net\nOurs†\nOurs‡ | 0.5 | 0.898\n0.906\n0.944 | 0.904\n0.903\n0.904 | 0.876\n0.880\n0.894 | 0.900\n0.889\n0.918 |\n| TS-Net\nOurs†\nOurs‡ | 0.6 | 0.848\n0.892\n0.920 | 0.886\n0.903\n0.904 | 0.864\n0.878\n0.894 | 0.871\n0.889\n0.918 |\n| TS-Net\nOurs†\nOurs‡ | 0.7 | 0.704\n0.802\n0.868 | 0.720\n0.820\n0.852 | 0.682\n0.746\n0.823 | 0.722\n0.797\n0.839 |\n| TS-Net\nOurs†\nOurs‡ | 0.8 | 0.496\n0.561\n0.680 | 0.597\n0.675\n0.748 | 0.565\n0.637\n0.714 | 0.582\n0.659\n0.735 |\n| TS-Net\nOurs†\nOurs‡ | 0.9 | 0.120\n0.325\n0.404 | 0.292\n0.307\n0.454 | 0.255\n0.296\n0.368 | 0.289\n0.301\n0.408 |'

In [55]:
tables_df[2]

,Method,IoU,FinTabNet,ICDAR-13,Sci-TSR,TUCD
0,None,None,TSR-F1↑,TSR-F1↑,TSR-F1↑,TSR-F1↑
1,TS-Net\nOurs†\nOurs‡,0.5,0.898\n0.906\n0.944,0.904\n0.903\n0.904,0.876\n0.880\n0.894,0.900\n0.889\n0.918
2,TS-Net\nOurs†\nOurs‡,0.6,0.848\n0.892\n0.920,0.886\n0.903\n0.904,0.864\n0.878\n0.894,0.871\n0.889\n0.918
3,TS-Net\nOurs†\nOurs‡,0.7,0.704\n0.802\n0.868,0.720\n0.820\n0.852,0.682\n0.746\n0.823,0.722\n0.797\n0.839
4,TS-Net\nOurs†\nOurs‡,0.8,0.496\n0.561\n0.680,0.597\n0.675\n0.748,0.565\n0.637\n0.714,0.582\n0.659\n0.735
5,TS-Net\nOurs†\nOurs‡,0.9,0.120\n0.325\n0.404,0.292\n0.307\n0.454,0.255\n0.296\n0.368,0.289\n0.301\n0.408


In [56]:
unstr_loader.max_chunk_size = 1000

In [57]:
chunks = unstr_loader.chunk_text(text[0], overlap = 100)

In [58]:
len(chunks)

52

In [59]:
len(chunks[0])

1000

## Embedding and Retreival

In [60]:
%pip install -q -U "google-genai>=1.0.0"

In [44]:
!pip install python_dotenv

In [61]:
from dotenv import load_dotenv

load_dotenv("config.env")

True

In [62]:
from google import genai
import os

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [63]:
MODEL_ID = "gemini-embedding-001"

In [64]:
result = client.models.embed_content(model = MODEL_ID, contents = chunks)

In [65]:
dim = len(result.embeddings[0].values)

In [66]:
dim

3072

In [67]:
import numpy as np
embeddings = np.empty((len(chunks), dim))

In [68]:
for i, embedding in enumerate(result.embeddings):
  embeddings[i] = np.array(embedding.values)

In [69]:
embeddings.shape

(52, 3072)

In [70]:
from util import FaissRetriever

In [71]:
retriever = FaissRetriever(embedding_dim=dim)

In [72]:
retriever.add(embeddings, chunks)

In [73]:
query = "TSR-Net"
query_embed = client.models.embed_content(model = MODEL_ID, contents = [query])
query_embedding = np.array(query_embed.embeddings[0].values).reshape(-1, dim)

In [74]:
retriever.retrieve(query_embedding)

['hat are far\napart in the two-dimensional space. To handle this prob-\nlem, we propose TSR-Net for structure recognition which\nuses the existing DGCNN architecture [22]. Our formulation\nuses rectilinear adjacencies instead of row/column adjacen-\ncies [22, 24]. Recursive parsing of rectilinear adjacencies\nhelps to build better long-range visual row/column associa-\ntions.\nOur contributions can be summarized as follows:\n• Introduce channel attention [19] for table object detec-\ntion and define two additional regularizers — conti-\nnuity and overlapping loss between every pair of cells\nin addition to the alignment loss from [24]. We use\ntrainable loss-weights for these losses and formulate a\nmin-max optimization problem for faster convergence.\n• Formulate structure recognition using rectilinear adja-\ncencies instead of row/column adjacencies, eliminat-\ning the need for complex post-processing heuristics for\ngenerating row and column spanning information for\nevery cell.\n•